# RevSplit Multi-Agent Training & Benchmark Evaluation Framework

This notebook evaluates, benchmarks, and fine-tunes the autonomous **Orchestrator-Worker Multi-Agent System** for RevSplit.

### Agent Roster:
1. **Supervisor Orchestrator**: Dynamic routing & step-limit loop capping.
2. **Agent A (Merchant Intelligence)**: Margin optimization & promo code extraction.
3. **Agent B (Fraud & Risk Scoring)**: Hybrid rule-based + Gemini LLM risk evaluation.
4. **Agent C (Personalization & Notification)**: High-converting markdown deal alert generation.

In [1]:
import os
import time
import json
import datetime
import certifi
from dotenv import load_dotenv

os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
print(f"Google Gemini API Key Configured: {bool(GOOGLE_API_KEY)}")

## 1. Import LangGraph & RevSplit Agent Modules

In [2]:
import sys
sys.path.append("..")

from app.agents.graph import agent_graph
from app.agents.tools import evaluate_transaction_risk_rules, fetch_merchant_margin_data, format_user_deal_alert

## 2. Test & Benchmark Unit Tool Executions

In [3]:
# Test Tool 1: Fraud Risk Rules
risk_res = evaluate_transaction_risk_rules.invoke({
    "sale_amount": 75000.0,
    "click_to_conversion_seconds": 2,
    "user_ip_country": "RUS"
})
print("--- Agent B Tool Result ---")
print(json.dumps(json.loads(risk_res), indent=2))

# Test Tool 2: Merchant Intelligence
intel_res = fetch_merchant_margin_data.invoke({"merchant_slug": "amazon"})
print("\n--- Agent A Tool Result ---")
print(json.dumps(json.loads(intel_res), indent=2))

## 3. Benchmark End-to-End Multi-Agent Orchestration Performance

In [4]:
test_cases = [
    {
        "name": "Normal User Purchase",
        "input": {
            "user_id": "usr_norm_101",
            "sub_id": "clk_101",
            "merchant_slug": "amazon",
            "gross_sale_amount": 1200.00,
            "click_to_conversion_seconds": 45,
            "user_ip_country": "IND",
            "risk_score": None,
            "risk_flagged": False,
            "risk_reasons": [],
            "best_margin_path": None,
            "recommended_deals": [],
            "personalized_message_md": None,
            "next_step": "supervisor",
            "step_count": 0,
            "max_steps": 6,
            "messages": [],
            "execution_trace": []
        }
    },
    {
        "name": "High Risk Velocity Anomaly",
        "input": {
            "user_id": "usr_fraud_999",
            "sub_id": "clk_999",
            "merchant_slug": "flipkart",
            "gross_sale_amount": 85000.00,
            "click_to_conversion_seconds": 1,
            "user_ip_country": "UKR",
            "risk_score": None,
            "risk_flagged": False,
            "risk_reasons": [],
            "best_margin_path": None,
            "recommended_deals": [],
            "personalized_message_md": None,
            "next_step": "supervisor",
            "step_count": 0,
            "max_steps": 6,
            "messages": [],
            "execution_trace": []
        }
    }
]

for tc in test_cases:
    print(f"\n=========================================")
    print(f"RUNNING EVALUATION TEST: {tc['name']}")
    print(f"=========================================")
    start_time = time.time()
    
    config = {"configurable": {"thread_id": f"eval_thread_{tc['name']}"}}
    res = agent_graph.invoke(tc["input"], config=config)
    
    elapsed = time.time() - start_time
    print(f"⚡ Total Latency: {elapsed:.3f} seconds")
    print(f"🛡 Risk Score: {res.get('risk_score')} (Flagged: {res.get('risk_flagged')})")
    print(f"📋 Steps Executed: {res.get('step_count')}")
    print(f"\n--- Execution Trace History ---")
    for tr in res.get("execution_trace", []):
        print(f"  Step {tr['step']} | {tr['agent_name']} -> {tr['action_taken']}")